# 06 - Baselines and Evaluation

We compare a PD controller and a linearized LQR baseline on the synthetic reference.


In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import yaml

ROOT = Path(r"e:/Optimal_Control/PINN/hjb_pinn_exoskeleton")
sys.path.append(str(ROOT))

from src.knee_dynamics import ExoParams, knee_dynamics_numpy
from src.baselines import linearize_knee, dlqr, pd_controller
from src.utils import sanitize_exo_params

cfg = yaml.safe_load((ROOT / "configs" / "knee_config.yaml").read_text())
params = ExoParams(**cfg["exo_params"])
params = sanitize_exo_params(params, cfg)


In [2]:
import pickle

# Build reference trajectory: use EPIC mean if available, else sinusoid
epic_path = ROOT / 'data' / 'epic_clean.pkl'
use_epic = epic_path.exists()

if use_epic:
    with epic_path.open('rb') as f:
        epic_samples = pickle.load(f)
    theta_mat = np.stack([s.theta for s in epic_samples])
    t_ref = epic_samples[0].time
    q_ref = theta_mat.mean(axis=0)
    dt = float(t_ref[1] - t_ref[0])
    qd_ref = np.gradient(q_ref, dt)
    qdd_ref = np.gradient(qd_ref, dt)
    print("Using EPIC mean reference")
else:
    # Sinusoidal fallback
    t_ref = np.linspace(0.0, 1.0, 501)
    q_ref = 0.6 * np.sin(2 * np.pi * t_ref)
    qd_ref = 0.6 * 2 * np.pi * np.cos(2 * np.pi * t_ref)
    qdd_ref = -0.6 * (2 * np.pi) ** 2 * np.sin(2 * np.pi * t_ref)
    print("Using sinusoidal reference")


Using EPIC mean reference


In [3]:
def simulate_controller(u_fn, params, t_ref, q_ref, qd_ref, qdd_ref, dt=0.002):
    steps = len(t_ref) - 1
    x = np.array([q_ref[0], qd_ref[0]], dtype=float)
    xs, us = [], []
    for k in range(steps):
        u = u_fn(x, q_ref[k], qd_ref[k], qdd_ref[k])
        dx = knee_dynamics_numpy(x, u, params)
        x = x + dt * dx
        xs.append(x.copy())
        us.append(u)
    return t_ref[:-1], np.array(xs), np.array(us)

# Resample reference to match dt
T = 1.0
dt = 0.002
t_sim = np.linspace(0.0, T, int(T/dt) + 1)
q_ref_sim = np.interp(t_sim, t_ref, q_ref)
qd_ref_sim = np.interp(t_sim, t_ref, qd_ref)
qdd_ref_sim = np.interp(t_sim, t_ref, qdd_ref)


In [4]:
import torch
from src.hjb_pinn import ValueNet, HJBConfig, compute_grads, optimal_torque, compute_control
from src.knee_dynamics import knee_dynamics_torch

# Proposed method: HJB-PINN policy (load from checkpoint if available)
ckpt_path = ROOT / "checkpoints" / "hjb_pinn_knee.pt"

model_cfg = cfg["model"]
weights = cfg["cost_weights"]
control = cfg.get("control", {})

cfg_hjb = HJBConfig(
    w_track=float(weights["w_track"]),
    w_omega=float(weights["w_omega"]),
    w_u=float(weights["w_u"]),
)

net = ValueNet(
    hidden_layers=int(model_cfg["hidden_layers"]),
    hidden_units=int(model_cfg["hidden_units"]),
    activation=str(model_cfg["activation"]),
)

ckpt_loaded = ckpt_path.exists()
ckpt_epoch = None
if ckpt_loaded:
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        net.load_state_dict(ckpt['state_dict'])
        ckpt_epoch = ckpt.get('epoch')
    else:
        net.load_state_dict(ckpt)
    net.eval()
    print("Loaded HJB-PINN checkpoint:", ckpt_path)
else:
    net = None
    print("HJB-PINN checkpoint not found. Train in 05_hjb_pinn_knee_training.ipynb and save to:")
    print(ckpt_path)


colloc_cfg = cfg.get("collocation", {})
q_scale = max(abs(float(colloc_cfg.get("theta_min", -1.0))), abs(float(colloc_cfg.get("theta_max", 1.0))), 1e-6)
qd_scale = max(abs(float(colloc_cfg.get("omega_min", -3.0))), abs(float(colloc_cfg.get("omega_max", 3.0))), 1e-6)

def V_eval(net, q_in, qd_in, t_in):
    return net(q_in / q_scale, qd_in / qd_scale, t_in)


def V_eval_with_grads(net, q_in, qd_in, t_in):
    qn = (q_in / q_scale).detach().requires_grad_(True)
    qdn = (qd_in / qd_scale).detach().requires_grad_(True)
    V = net(qn, qdn, t_in)
    V_t, V_qn, V_qdn = compute_grads(V, qn, qdn, t_in)
    V_q = V_qn / q_scale
    V_qd = V_qdn / qd_scale
    return V, V_t, V_q, V_qd


def simulate_hjb_pinn(net, params, T=1.0, dt=0.002, q0=0.0, qd0=0.0):
    steps = int(T / dt)
    t = np.linspace(0.0, T, steps + 1, dtype=np.float32)
    x = torch.tensor([[q0, qd0]], dtype=torch.float32, requires_grad=True)
    u_max = float(control.get("u_max", 20.0))
    use_tanh = bool(control.get("use_tanh_saturation", False))
    sat_method = "tanh" if use_tanh else "clamp"
    u_scale = float(control.get("u_scale", 1.0))
    cap_multiplier = bool(control.get("cap_multiplier", True))
    multiplier_cap = float(control.get("multiplier_cap", 1e3))

    xs = []
    us = []
    for k in range(steps):
        q = x[:, 0:1]
        qd = x[:, 1:2]
        tt = torch.tensor([[t[k]]], dtype=torch.float32, requires_grad=True)
        V, _, _, V_qd = V_eval_with_grads(net, q, qd, tt)
        _, u = compute_control(V_qd, params, cfg_hjb.w_u, u_max=u_max, saturation_method=sat_method, u_scale=u_scale, cap_multiplier=cap_multiplier, multiplier_cap=multiplier_cap)
        dx = knee_dynamics_torch(x, u, params)
        x = (x + dt * dx).detach().requires_grad_(True)

        xs.append(x.detach().numpy()[0])
        u_val = u.detach().cpu().numpy().squeeze()
        us.append(float(u_val))

    return t[:-1], np.array(xs), np.array(us)


Loaded HJB-PINN checkpoint: e:\Optimal_Control\PINN\hjb_pinn_exoskeleton\checkpoints\hjb_pinn_knee.pt


In [5]:
# PD comparison vs learned HJB-PINN
if net is not None:
    u_max = float(control.get("u_max", 20.0))
    base_cfg = cfg.get("baseline", {})
    kp = float(base_cfg.get("Kp", 30.0))
    kd = float(base_cfg.get("Kd", 5.0))

    def u_pd_fn(x, q_ref_k, qd_ref_k, qdd_ref_k):
        u = pd_controller(x[0], x[1], q_ref_k, qd_ref_k, kp, kd)
        return float(np.clip(u, -u_max, u_max))

    t_pd, x_pd, u_pd = simulate_controller(u_pd_fn, params, t_sim, q_ref_sim, qd_ref_sim, qdd_ref_sim, dt=dt)
    t_hjb, x_hjb, u_hjb = simulate_hjb_pinn(net, params, T=T, dt=dt, q0=q_ref_sim[0], qd0=qd_ref_sim[0])

    plt.figure(figsize=(10, 4))
    plt.plot(t_sim[:-1], q_ref_sim[:-1], label="q_ref")
    plt.plot(t_hjb, x_hjb[:, 0], label="q (HJB)")
    plt.plot(t_pd, x_pd[:, 0], label="q (PD)")
    plt.title("Tracking: HJB vs PD")
    plt.xlabel("t")
    plt.ylabel("angle (rad)")
    plt.legend()
    plt.grid(False)
    plt.show()

    plt.figure(figsize=(10, 3))
    plt.plot(t_hjb, u_hjb, label="u (HJB)")
    plt.plot(t_pd, u_pd, label="u (PD)")
    plt.title("Torque: HJB vs PD")
    plt.xlabel("t")
    plt.ylabel("u (Nm)")
    plt.legend()
    plt.grid(False)
    plt.show()

    rmse_hjb = float(np.sqrt(np.mean((x_hjb[:, 0] - q_ref_sim[: len(x_hjb)]) ** 2)))
    rmse_pd = float(np.sqrt(np.mean((x_pd[:, 0] - q_ref_sim[: len(x_pd)]) ** 2)))
    print(f"RMSE HJB: {rmse_hjb:.4f} | RMSE PD: {rmse_pd:.4f}")

# Debug summary
if net is not None:
    q_ref_plot = q_ref_sim if 'q_ref_sim' in globals() else q_ref
    print("q_ref range:", float(np.min(q_ref_plot)), float(np.max(q_ref_plot)))
    if 'x_hjb' in globals():
        print("q range:", float(np.min(x_hjb[:, 0])), float(np.max(x_hjb[:, 0])))
    else:
        print("q range: <run simulate_hjb_pinn cell>")
    if 'u_hjb' in globals():
        print("u range:", float(np.min(u_hjb)), float(np.max(u_hjb)))
    else:
        print("u range: <run simulate_hjb_pinn cell>")
    if 'x_pd' in globals():
        print("q (PD) range:", float(np.min(x_pd[:, 0])), float(np.max(x_pd[:, 0])))
    if 'u_pd' in globals():
        print("u (PD) range:", float(np.min(u_pd)), float(np.max(u_pd)))
    print("checkpoint loaded:", ckpt_loaded, "epoch:", ckpt_epoch)


q_ref range: -0.7391093959324314 -0.01824299432337284


NameError: name 'x_hjb' is not defined